# Volume-Level Evaluation For All Runs

This notebook runs paper-style volume-level evaluation for every training run in `runs/`.

For 2D networks, it calls `scripts/evaluate_2d.py`, which predicts every 2D slice and reconstructs a 3D patient/frame prediction volume before computing metrics. For the 3D network, it calls `scripts/evaluate_3d.py` directly on 3D HDF5 volumes.

It then compares all evaluated runs with each other and with Dice values reported in *An Exploration of 2D and 3D Deep Learning Techniques for Cardiac MR Image Segmentation*.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.precision", 4)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNS_ROOT = PROJECT_ROOT / "runs"

# Evaluation controls. Set FORCE_RERUN=True when you want to recompute even if CSVs already exist.
EVAL_SPLIT = "val"  # "val", "train", or "all"
FORCE_RERUN = False
MAX_VOLUMES = None  # use an integer for a quick smoke test, e.g. 2
BATCH_SIZE_2D = 8

print(f"Project root: {PROJECT_ROOT}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Python: {sys.executable}")


## Paper Reference Results

The paper reports Dice, ASSD, and Hausdorff distance by structure and cardiac phase. For a compact comparison against local volume-level evaluation, this notebook averages the paper's ED and ES Dice for each model/structure.


In [ ]:
paper_architecture_values = {
    "FCN-8": {
        "LV": {"ED": 0.960, "ES": 0.926},
        "RV": {"ED": 0.932, "ES": 0.835},
        "Myo": {"ED": 0.869, "ES": 0.890},
    },
    "2D U-Net": {
        "LV": {"ED": 0.965, "ES": 0.937},
        "RV": {"ED": 0.936, "ES": 0.838},
        "Myo": {"ED": 0.885, "ES": 0.904},
    },
    "2D U-Net (mod.)": {
        "LV": {"ED": 0.966, "ES": 0.935},
        "RV": {"ED": 0.934, "ES": 0.852},
        "Myo": {"ED": 0.892, "ES": 0.906},
    },
    "3D U-Net (mod.)": {
        "LV": {"ED": 0.939, "ES": 0.905},
        "RV": {"ED": 0.888, "ES": 0.781},
        "Myo": {"ED": 0.802, "ES": 0.839},
    },
}

paper_rows = []
for model, structures in paper_architecture_values.items():
    for structure, phases in structures.items():
        for phase, dice in phases.items():
            paper_rows.append({"paper_model": model, "class_name": structure, "phase": phase, "paper_dice": dice})

paper_phase = pd.DataFrame(paper_rows)
paper_by_class = paper_phase.groupby(["paper_model", "class_name"], as_index=False).agg(
    paper_phase_avg_dice=("paper_dice", "mean"),
    paper_ed_dice=("paper_dice", lambda s: float(s.iloc[0])),
)
paper_model_mean = paper_by_class.groupby("paper_model", as_index=False).agg(
    paper_mean_fg_dice=("paper_phase_avg_dice", "mean")
)

display(paper_by_class)
display(paper_model_mean.sort_values("paper_mean_fg_dice", ascending=False))


## Discover Runs And Checkpoints

In [ ]:
def read_json(path):
    return json.loads(path.read_text()) if path.exists() else {}


def config_args(config):
    return config.get("args", {})


def infer_model_key(run_name, config):
    text = f"{run_name} {config_args(config).get('run_dir', '')}".lower()
    if "fcn8" in text:
        return "fcn8"
    if "unet3d" in text:
        return "unet3d"
    if "unet2d_modified" in text or "modified" in text:
        return "unet2d_modified"
    if "unet2d" in text:
        return "unet2d"
    return "unknown"


def paper_model_name(model_key):
    return {
        "fcn8": "FCN-8",
        "unet2d": "2D U-Net",
        "unet2d_modified": "2D U-Net (mod.)",
        "unet3d": "3D U-Net (mod.)",
    }.get(model_key, "unknown")


def find_checkpoint(run_dir):
    for pattern in ["best_epoch_*.pt", "latest_epoch_*.pt", "*.pt"]:
        matches = sorted(run_dir.glob(pattern))
        if matches:
            return matches[-1]
    return None


def default_eval_output_dir(run_dir, model_key, split):
    kind = "3d" if model_key == "unet3d" else "2d"
    return run_dir / f"evaluation_{kind}_{split}"


run_rows = []
for run_dir in sorted(path for path in RUNS_ROOT.iterdir() if path.is_dir()):
    config = read_json(run_dir / "config.json")
    model_key = infer_model_key(run_dir.name, config)
    checkpoint = find_checkpoint(run_dir)
    data_dir = config_args(config).get("data_dir")
    eval_dir = default_eval_output_dir(run_dir, model_key, EVAL_SPLIT) if model_key != "unknown" else None
    run_rows.append({
        "run": run_dir.name,
        "run_dir": run_dir,
        "model_key": model_key,
        "paper_model": paper_model_name(model_key),
        "checkpoint": checkpoint,
        "has_checkpoint": checkpoint is not None,
        "data_dir_from_config": data_dir,
        "eval_dir": eval_dir,
        "metrics_csv": None if eval_dir is None else eval_dir / "metrics_by_class.csv",
        "summary_json": None if eval_dir is None else eval_dir / "summary.json",
    })

runs = pd.DataFrame(run_rows)
display(runs[["run", "model_key", "paper_model", "has_checkpoint", "checkpoint", "data_dir_from_config", "eval_dir"]])


## Run Evaluations

This cell executes the appropriate evaluator for each run that has a checkpoint. Existing evaluation CSVs are reused unless `FORCE_RERUN=True`.


In [ ]:
def run_evaluation(row):
    if row.model_key == "unknown":
        return {"run": row.run, "status": "skipped", "reason": "unknown model type"}
    if not row.has_checkpoint:
        return {"run": row.run, "status": "skipped", "reason": "no checkpoint .pt file found"}
    if row.metrics_csv is not None and Path(row.metrics_csv).exists() and not FORCE_RERUN:
        return {"run": row.run, "status": "cached", "reason": "metrics_by_class.csv already exists"}

    if row.model_key == "unet3d":
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "evaluate_3d.py"),
            "--run-dir", str(row.run_dir),
            "--split", EVAL_SPLIT,
        ]
    else:
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "evaluate_2d.py"),
            "--run-dir", str(row.run_dir),
            "--model", row.model_key,
            "--split", EVAL_SPLIT,
            "--batch-size", str(BATCH_SIZE_2D),
        ]

    if MAX_VOLUMES is not None:
        command += ["--max-volumes", str(MAX_VOLUMES)]

    result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    status = "ok" if result.returncode == 0 else "failed"
    return {
        "run": row.run,
        "status": status,
        "reason": "" if result.returncode == 0 else result.stderr[-2000:],
        "stdout_tail": result.stdout[-2000:],
        "command": " ".join(command),
    }


eval_results = pd.DataFrame([run_evaluation(row) for row in runs.itertuples()])
display(eval_results[["run", "status", "reason"]])

failed = eval_results[eval_results["status"] == "failed"]
if not failed.empty:
    print("Failed commands:")
    display(failed[["run", "command", "reason"]])


## Load Evaluation Outputs

In [ ]:
metric_frames = []
summary_rows = []

for row in runs.itertuples():
    metrics_path = Path(row.metrics_csv) if row.metrics_csv is not None else None
    summary_path = Path(row.summary_json) if row.summary_json is not None else None
    if metrics_path is not None and metrics_path.exists():
        df = pd.read_csv(metrics_path)
        df["run"] = row.run
        df["model_key"] = row.model_key
        df["paper_model"] = row.paper_model
        df["metrics_path"] = metrics_path
        metric_frames.append(df)
    if summary_path is not None and summary_path.exists():
        payload = json.loads(summary_path.read_text())
        summary = payload.get("summary", {})
        metadata = payload.get("metadata", {})
        summary_rows.append({
            "run": row.run,
            "model_key": row.model_key,
            "paper_model": row.paper_model,
            **summary,
            "checkpoint": metadata.get("checkpoint"),
            "data_dir": metadata.get("data_dir"),
            "volumes_evaluated": metadata.get("volumes_evaluated"),
        })

all_metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
run_summary = pd.DataFrame(summary_rows)

print(f"Evaluation metric rows loaded: {len(all_metrics)}")
print(f"Run summaries loaded: {len(run_summary)}")
if not run_summary.empty:
    display(run_summary.sort_values("mean_dice", ascending=False))
else:
    print("No evaluation outputs found yet. Runs without checkpoints cannot be evaluated.")


## Compare Runs With Each Other

In [ ]:
if all_metrics.empty:
    print("No evaluation metrics available for comparison.")
else:
    by_run_class = all_metrics.groupby(["run", "paper_model", "class_name"], as_index=False).agg(
        dice=("dice", "mean"),
        assd_mm=("assd_mm", "mean"),
        hd_mm=("hd_mm", "mean"),
        volumes=("patient", "nunique"),
    )
    display(by_run_class.sort_values(["run", "class_name"]))

    plt.figure(figsize=(max(10, 0.7 * by_run_class["run"].nunique()), 5))
    sns.barplot(data=by_run_class, x="run", y="dice", hue="class_name")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.title("Volume-Level Dice By Run And Structure")
    plt.tight_layout()

    if not run_summary.empty:
        plt.figure(figsize=(max(10, 0.7 * len(run_summary)), 5))
        order = run_summary.sort_values("mean_dice", ascending=False)["run"]
        sns.barplot(data=run_summary, x="run", y="mean_dice", order=order)
        plt.xticks(rotation=45, ha="right")
        plt.ylim(0, 1)
        plt.title("Mean Foreground Dice By Run")
        plt.tight_layout()


## Compare With Paper Results

In [ ]:
if all_metrics.empty:
    print("No evaluation metrics available for paper comparison.")
else:
    by_run_class = all_metrics.groupby(["run", "paper_model", "class_name"], as_index=False).agg(
        local_dice=("dice", "mean"),
        local_assd_mm=("assd_mm", "mean"),
        local_hd_mm=("hd_mm", "mean"),
    )
    comparison = by_run_class.merge(paper_by_class, on=["paper_model", "class_name"], how="left")
    comparison["delta_vs_paper_phase_avg"] = comparison["local_dice"] - comparison["paper_phase_avg_dice"]
    display(comparison.sort_values(["paper_model", "run", "class_name"]).style.format({
        "local_dice": "{:.3f}",
        "paper_phase_avg_dice": "{:.3f}",
        "delta_vs_paper_phase_avg": "{:+.3f}",
        "local_assd_mm": "{:.2f}",
        "local_hd_mm": "{:.2f}",
    }))

    plot_data = comparison.dropna(subset=["paper_phase_avg_dice"])
    if not plot_data.empty:
        plt.figure(figsize=(max(10, 0.8 * plot_data["run"].nunique()), 5))
        sns.barplot(data=plot_data, x="run", y="delta_vs_paper_phase_avg", hue="class_name")
        plt.axhline(0, color="black", linewidth=1)
        plt.xticks(rotation=45, ha="right")
        plt.title("Local Volume Dice Minus Paper ED/ES-Averaged Dice")
        plt.ylabel("Dice difference")
        plt.tight_layout()

    model_summary = run_summary.merge(paper_model_mean, on="paper_model", how="left") if not run_summary.empty else pd.DataFrame()
    if not model_summary.empty:
        model_summary["delta_vs_paper_mean_fg"] = model_summary["mean_dice"] - model_summary["paper_mean_fg_dice"]
        display(model_summary[["run", "paper_model", "mean_dice", "paper_mean_fg_dice", "delta_vs_paper_mean_fg", "mean_assd_mm", "mean_hd_mm"]].sort_values("mean_dice", ascending=False).style.format({
            "mean_dice": "{:.3f}",
            "paper_mean_fg_dice": "{:.3f}",
            "delta_vs_paper_mean_fg": "{:+.3f}",
            "mean_assd_mm": "{:.2f}",
            "mean_hd_mm": "{:.2f}",
        }))


## Skipped Or Failed Runs

In [ ]:
if eval_results.empty:
    print("No evaluation attempts recorded.")
else:
    skipped_or_failed = eval_results[eval_results["status"].isin(["skipped", "failed"])]
    if skipped_or_failed.empty:
        print("No skipped or failed runs.")
    else:
        display(skipped_or_failed[["run", "status", "reason"]])


## Notes

- This notebook compares local evaluation metrics against the paper's published Dice values. Local metrics are only directly comparable if preprocessing, split, postprocessing, and evaluation space match the paper.
- The current evaluator reconstructs/evaluates in the preprocessed HDF5 space. It uses spacing metadata for ASSD and Hausdorff distance in millimetres, but it does not yet warp predictions back to the original NIfTI grid.
- Runs must have saved `.pt` checkpoints. A run folder with only `metrics.csv` and `config.json` cannot be re-evaluated.
